test notebook. 
attempting to visualize urdf


In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
# Native urdfpy: use URDF next to meshes with relative paths (see urdfpy URDF.load docstring).
from pathlib import Path
import numpy as np
from urdfpy import URDF

root = Path(r"C:/Users/dnxjc/Desktop/CURR/PROJECTS/rewind")
urdf_path = root / "rewind_glove_assembly/urdf/rewind_glove_for_urdfpy.urdf"
robot = URDF.load(str(urdf_path))

# If you must load the OnShape URDF with package://mesh URIs instead, use:
#   sys.path.insert(0, str(root / "glove_sim"))
#   from src.urdfpy_vis import load_robot
#   robot = load_robot(root / "rewind_glove_assembly/urdf/rewind_glove_assembly.urdf", root / "rewind_glove_assembly/meshes")

In [2]:
# 2. Visualize the glove in its rest state
print("Visualizing rest pose. Close the window to continue...")
robot.show()

Visualizing rest pose. Close the window to continue...


In [4]:
# 3. Animate joints. If live viewer fails (OpenGL driver issue), export GLB frames instead.
import trimesh

traj = {
    "revolute_3_0": [np.deg2rad(0), np.deg2rad(4)],
    "revolute_9_0": [np.deg2rad(0), np.deg2rad(4)],
}

print("Attempting live animation window...")
try:
    robot.animate(cfg_trajectory=traj)
    print("Done!")
except Exception as e:
    print("Live animation unavailable:", type(e).__name__, e)
    print("Falling back to GLB frame export...")

    out_dir = root / "glove_sim_v2/outputs/animate_frames"
    out_dir.mkdir(parents=True, exist_ok=True)

    n_frames = 12
    for i in range(n_frames):
        t = i / (n_frames - 1)
        cfg = {
            "revolute_3_0": (1 - t) * traj["revolute_3_0"][0] + t * traj["revolute_3_0"][1],
            "revolute_9_0": (1 - t) * traj["revolute_9_0"][0] + t * traj["revolute_9_0"][1],
        }
        fk = robot.visual_trimesh_fk(cfg=cfg)
        scene = trimesh.Scene()
        for mesh, pose in fk.items():
            m = mesh.copy()
            m.apply_transform(pose)
            scene.add_geometry(m)
        out_path = out_dir / f"frame_{i:03d}.glb"
        scene.export(str(out_path))

    print(f"Exported {n_frames} GLB frames to: {out_dir}")
    print("Open frame_000.glb and frame_011.glb (or scrub all frames) in f3d/Blender.")

Exception in thread Thread-10 (_init_and_start_app):
Traceback (most recent call last):
  File "c:\Users\dnxjc\miniforge3\envs\glove_sim\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "c:\Users\dnxjc\miniforge3\envs\glove_sim\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\dnxjc\miniforge3\envs\glove_sim\lib\site-packages\pyrender\viewer.py", line 1016, in _init_and_start_app
    super(Viewer, self).__init__(config=conf, resizable=True,
  File "c:\Users\dnxjc\miniforge3\envs\glove_sim\lib\site-packages\pyglet\window\win32\__init__.py", line 165, in __init__
    super().__init__(*args, **kwargs)
  File "c:\Users\dnxjc\miniforge3\envs\glove_sim\lib\site-packages\pyglet\window\__init__.py", line 545, in __init__
    config = screen.get_best_config(config)
  File "c:\Users\dnxjc\miniforge3\envs\glove_sim\lib\site-packages\pyglet\display\base.py", line 129, in get_best_config
    configs = self.get_matching_configs(

Attempting live animation window...


KeyboardInterrupt: 

### Why the raw OnShape / ROS URDF broke `URDF.load`

1. **`package://pkg/meshes/file.stl`** — ROS resolves that via package search paths. **urdfpy does not**; it uses `os.path.join(urdf_directory, filename)` for non-absolute paths, so you get a bogus path that still contains `package://…`.
2. **What upstream expects** — Same as `urdfpy.utils.get_filename`: mesh `filename` should be **relative to the `.urdf` file** (or an absolute path). The companion `rewind_glove_for_urdfpy.urdf` uses `../meshes/...` for that reason.
3. **`load_robot`** — Rewrites `package://` to a path **relative to the URDF folder** (using your `mesh_dir` to compute `relpath`), strips invalid empty `<texture/>` tags, writes a temp `.urdf` **next to** the real one, then calls **`URDF.load`** — same parser path as “native” urdfpy.

### `glove_sim` / MuJoCo

MuJoCo builds MJCF from the same URDF mesh attributes: it joins **`cfg.MESH_DIR`** with the attribute (after stripping `package://` to a basename, or normalizing `../meshes/...`). Use **binary STLs** under `glove_sim/assets/meshes` (`convert_meshes_for_mujoco.py`) so MuJoCo’s loader is happy; urdfpy can still load from `rewind_glove_assembly/meshes` (trimesh accepts ASCII STL).

In [5]:
# load_robot: same urdfpy URDF.load path after fixing package:// + textures
import sys
from pathlib import Path

root = Path(r"C:/Users/dnxjc/Desktop/CURR/PROJECTS/rewind")
sys.path.insert(0, str(root / "glove_sim"))

from src.urdfpy_vis import load_robot

canonical_urdf = root / "rewind_glove_assembly/urdf/rewind_glove_assembly.urdf"
assembly_mesh_dir = root / "rewind_glove_assembly/meshes"

robot_via_helper = load_robot(canonical_urdf, assembly_mesh_dir)
print("load_robot(canonical URDF, assembly/meshes):", len(robot_via_helper.links), "links")
print("  actuated joints:", len(robot_via_helper.actuated_joints))

assert len(robot_via_helper.links) == len(robot.links), "should match native `robot` from cell 1"
assert {j.name for j in robot_via_helper.actuated_joints} == {j.name for j in robot.actuated_joints}
print("OK: link count and actuated joint names match native urdfpy load.")

load_robot(canonical URDF, assembly/meshes): 22 links
  actuated joints: 9
OK: link count and actuated joint names match native urdfpy load.


In [6]:
# Resolve one mesh the same way urdfpy vs MuJoCo pipeline do
import os
import xml.etree.ElementTree as ET
from pathlib import Path

root = Path(r"C:/Users/dnxjc/Desktop/CURR/PROJECTS/rewind")


def resolve_like_urdfpy(urdf_file: Path, mesh_filename_attr: str) -> Path:
    """urdfpy.utils.get_filename(urdf_dir, file_path)."""
    base = str(urdf_file.parent.resolve())
    fn = mesh_filename_attr
    if os.path.isabs(fn):
        return Path(fn)
    return Path(os.path.normpath(os.path.join(base, fn)))


def resolve_like_mujoco_glove_ik(mesh_dir: Path, mesh_filename_attr: str) -> Path:
    """glove_ik: strip package:// to basename chain, else keep attr; normpath(mesh_dir / fn)."""
    fn = mesh_filename_attr
    if fn.startswith("package://"):
        fn = fn.split("/", 2)[-1]
        fn = fn.split("/", 1)[-1]
        fn = fn.split("/", 1)[-1]
    return Path(os.path.normpath(str(mesh_dir.resolve() / fn)))

urdf_for_urdfpy = root / "rewind_glove_assembly/urdf/rewind_glove_for_urdfpy.urdf"
mesh_attr = next(m.get("filename") for m in ET.parse(urdf_for_urdfpy).getroot().iter("mesh"))

p_urdfpy = resolve_like_urdfpy(urdf_for_urdfpy, mesh_attr)
mujoco_mesh_dir = root / "glove_sim/assets/meshes"
p_mujoco = resolve_like_mujoco_glove_ik(mujoco_mesh_dir, mesh_attr)

print("Example <mesh filename=...>:", repr(mesh_attr))
print("  urdfpy (join URDF dir + attribute):", p_urdfpy, "| exists:", p_urdfpy.is_file())
print("  MuJoCo cfg.MESH_DIR join + normpath:", p_mujoco, "| exists:", p_mujoco.is_file())
if p_urdfpy.is_file() and p_mujoco.is_file():
    print("OK: visualization (assembly) and MuJoCo (binary copy) both find this STL.")
elif p_urdfpy.is_file() and not p_mujoco.is_file():
    print("Run from repo root: python glove_sim/convert_meshes_for_mujoco.py")

Example <mesh filename=...>: '../meshes/Hand Mount.stl'
  urdfpy (join URDF dir + attribute): C:\Users\dnxjc\Desktop\CURR\PROJECTS\rewind\rewind_glove_assembly\meshes\Hand Mount.stl | exists: True
  MuJoCo cfg.MESH_DIR join + normpath: C:\Users\dnxjc\Desktop\CURR\PROJECTS\rewind\glove_sim\assets\meshes\Hand Mount.stl | exists: True
OK: visualization (assembly) and MuJoCo (binary copy) both find this STL.


In [7]:
# MuJoCo: same URDF + cfg.MESH_DIR as pipeline / diagnostic
import sys
from pathlib import Path

root = Path(r"C:/Users/dnxjc/Desktop/CURR/PROJECTS/rewind")
sys.path.insert(0, str(root / "glove_sim"))

import config as cfg
from src.glove_ik import GloveSimulator

print("URDF:", cfg.URDF_PATH)
print("MESH_DIR:", cfg.MESH_DIR)
try:
    sim = GloveSimulator(cfg.URDF_PATH, cfg.MESH_DIR)
    print("MuJoCo GloveSimulator OK — nq=", sim.model.nq, "nv=", sim.model.nv, "nbody=", sim.model.nbody)
except Exception as e:
    print("MuJoCo load failed:", type(e).__name__, e)
    print("If missing/corrupt STLs under assets/meshes, run: python glove_sim/convert_meshes_for_mujoco.py")

URDF: C:\Users\dnxjc\Desktop\CURR\PROJECTS\rewind\rewind_glove_assembly\urdf\rewind_glove_assembly.urdf
MESH_DIR: C:\Users\dnxjc\Desktop\CURR\PROJECTS\rewind\glove_sim\assets\meshes
MuJoCo GloveSimulator OK — nq= 16 nv= 15 nbody= 22


### Visualize via updated `urdfpy_vis.get_glove_scene`

This uses the updated `load_robot`/`get_glove_scene` path (same FK path as urdfpy, mesh path fixes included) and exports GLBs you can open directly.

In [8]:
# Export rest + bent scenes from updated urdfpy_vis
import sys
from pathlib import Path
import numpy as np

root = Path(r"C:/Users/dnxjc/Desktop/CURR/PROJECTS/rewind")
sys.path.insert(0, str(root / "glove_sim"))

from src.urdfpy_vis import load_robot, get_glove_scene

canonical_urdf = root / "rewind_glove_assembly/urdf/rewind_glove_assembly.urdf"
assembly_mesh_dir = root / "rewind_glove_assembly/meshes"
robot_vis = load_robot(canonical_urdf, assembly_mesh_dir)

# Hand-mount world pose (identity) for quick inspection
T_hand_mount_world = np.eye(4, dtype=float)

# Build joint configs using all actuated joints, then override two for a visible bend
joint_cfg_rest = {j.name: 0.0 for j in robot_vis.actuated_joints}
joint_cfg_bent = dict(joint_cfg_rest)
joint_cfg_bent["revolute_3_0"] = np.deg2rad(45.0)
joint_cfg_bent["revolute_9_0"] = np.deg2rad(45.0)

scene_rest = get_glove_scene(robot_vis, joint_cfg_rest, T_hand_mount_world)
scene_bent = get_glove_scene(robot_vis, joint_cfg_bent, T_hand_mount_world)

out_dir = root / "glove_sim_v2/outputs/urdfpy_vis_scene"
out_dir.mkdir(parents=True, exist_ok=True)
out_rest = out_dir / "glove_rest_urdfpy_vis.glb"
out_bent = out_dir / "glove_bent_urdfpy_vis.glb"

scene_rest.export(str(out_rest))
scene_bent.export(str(out_bent))

print("Exported:")
print(" ", out_rest)
print(" ", out_bent)
print("Tip: open these in f3d/Blender to confirm links stay connected through joints.")

Exported:
  C:\Users\dnxjc\Desktop\CURR\PROJECTS\rewind\glove_sim_v2\outputs\urdfpy_vis_scene\glove_rest_urdfpy_vis.glb
  C:\Users\dnxjc\Desktop\CURR\PROJECTS\rewind\glove_sim_v2\outputs\urdfpy_vis_scene\glove_bent_urdfpy_vis.glb
Tip: open these in f3d/Blender to confirm links stay connected through joints.


In [9]:
# Optional: short sweep to verify continuity of linkage motion
sweep_dir = out_dir / "index_tip_sweep"
sweep_dir.mkdir(parents=True, exist_ok=True)

for deg in range(0, 91, 15):
    cfg = dict(joint_cfg_rest)
    cfg["revolute_9_0"] = np.deg2rad(float(deg))
    scene = get_glove_scene(robot_vis, cfg, T_hand_mount_world)
    scene.export(str(sweep_dir / f"index_{deg:03d}.glb"))

print(f"Exported sweep frames to: {sweep_dir}")

Exported sweep frames to: C:\Users\dnxjc\Desktop\CURR\PROJECTS\rewind\glove_sim_v2\outputs\urdfpy_vis_scene\index_tip_sweep
